In [0]:
import logging
from pyspark.sql.functions import col, current_timestamp, lit, when
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# Configure server-side console logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("SalesPipeline")

try:
    # ==========================================
    # STEP 1: SIMULATE INCOMING RAW DATA (Mocking Source Data)
    # ==========================================
    # Notice: Rows 1 & 2 are perfect. Row 3 has a NULL id. Row 4 has a '0' quantity anomaly.
    raw_data = [
        ("TXN101", "CUST450", 5),   # PASS
        ("TXN102", "CUST891", 12),  # PASS
        (None, "CUST202", 3),       # FAIL: Missing Primary Key (NULL)
        ("TXN104", "CUST311", 0)    # FAIL: Invalid business value (0 Quantity)
    ]
    
    schema = StructType([
        StructField("transactionID", StringType(), True),
        StructField("customerID", StringType(), True),
        StructField("quantity", IntegerType(), True)
    ])
    
    df_raw = spark.createDataFrame(raw_data, schema)
    logger.info("Step 1 Complete: Raw source data read successfully.")

    # ==========================================
    # STEP 2: ROW-LEVEL VALIDATION (The Split Logic)
    # ==========================================
    # Define our strict operational rules
    valid_condition = (col("transactionID").isNotNull()) & (col("quantity") > 0)
    
    # Isolate Clean Rows (Passes)
    df_pass = df_raw.filter(valid_condition) \
                    .withColumn("processed_at", current_timestamp())
                    
    # Isolate Bad Rows (Failures)
    df_fail = df_raw.filter(~valid_condition) \
                    .withColumn("rejection_reason", 
                        when(col("transactionID").isNull(), lit("Missing TransactionID"))
                        .when(col("quantity") <= 0, lit("Invalid Quantity"))
                        .otherwise(lit("Unknown Error"))
                    )

    # ==========================================
    # STEP 3: PERSIST CLEAN DATA (Eager Action 1)
    # ==========================================
    pass_count = df_pass.count()
    if pass_count > 0:
        df_pass.write \
               .format("delta") \
               .mode("append") \
               .saveAsTable("sales_gold")
        logger.info(f"Step 3 Complete: Successfully routed {pass_count} valid rows to Sales Gold table.")
    else:
        logger.warning("No clean rows found to insert into Sales Gold.")

    # ==========================================
    # STEP 4: ROUTE BAD DATA TO THE DEAD LETTER QUEUE (Eager Action 2)
    # ==========================================
    fail_count = df_fail.count()
    if fail_count > 0:
        df_fail.write \
               .format("delta") \
               .mode("append") \
               .saveAsTable("dead_letter_queue")
        logger.warning(f"Step 4 Complete: Isolated {fail_count} anomalies in the Dead Letter Queue.")
    
    print(f"--- Pipeline Execution Summary ---")
    print(f"Rows Saved to Production: {pass_count}")
    print(f"Rows Routed to DLQ: {fail_count}")

except Exception as system_error:
    # ==========================================
    # STEP 5: CRITICAL SYSTEM FAULT SAFEGUARD
    # ==========================================
    # This catches severe infrastructure breaks (e.g., cluster network drops, storage full)
    logger.error(f"CRITICAL PIPELINE HALT: Internal Cluster Failure. Details: {str(system_error)}")
    raise system_error
